# MINI Cells — Experiment 019b: Recruitment Response Curves

Checkpoint-only follow-up to stable Experiment 019. No Phase-1 or donor training is performed. The experiment scans finite recruitment amplitudes to test for activation barriers and finite probationary utility.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
ROOT = Path('/kaggle/working/mini-cells')
os.chdir('/kaggle/working')
if not (ROOT / '.git').exists():
    raise RuntimeError('019b requires the existing stable-019 Kaggle workspace because model checkpoints are intentionally not stored in GitHub. Reopen/use the workspace that produced stable 019.')
subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
subprocess.run(['git','switch','main'], cwd=ROOT, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=ROOT, check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[lm]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
subprocess.run(['git','rev-parse','HEAD'], check=True)


In [ ]:
import torch
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu_count': torch.cuda.device_count(), 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
if torch.cuda.device_count() < 1:
    raise RuntimeError('Experiment 019b requires a Kaggle GPU accelerator')


## Check stable-019 checkpoint source


In [ ]:
SOURCE = ROOT / 'results' / 'proposal-utility-discovery-stable-v1'
manifest_path = SOURCE / 'checkpoint-manifest.json'
decision_path = SOURCE / 'decision.json'
if not manifest_path.exists() or not decision_path.exists():
    raise RuntimeError('stable-019 local result/checkpoint source is missing')
manifest = json.loads(manifest_path.read_text())
decision019 = json.loads(decision_path.read_text())
print({'source_status': decision019['status'], 'checkpoint_files': manifest['file_count'], 'expected': manifest['expected_file_count']})
if decision019['status'] != 'UTILITY_ORACLE_INCONSISTENT' or manifest['file_count'] != manifest['expected_file_count']:
    raise RuntimeError('019b requires the complete stable-019 UTILITY_ORACLE_INCONSISTENT run')


## Preflight


In [ ]:
tests = [
    'tests/research/02-self-organization/test_language_recruitment_response.py',
    'tests/research/02-self-organization/test_language_recruitment_numerics.py',
    'tests/research/02-self-organization/test_language_proposal_checkpoints.py',
]
subprocess.run([sys.executable,'-m','pytest',*tests,'-q'], cwd=ROOT, check=True)


## Run checkpoint-only response sweep

Completed replicate response files are reused automatically. No donor training is possible from this entry point.


In [ ]:
OUT = ROOT / 'results' / 'recruitment-response-curves-v1'
subprocess.run([sys.executable,'scripts/research/run_recruitment_response_curves.py'], cwd=ROOT, check=True)
print('results:', OUT)


In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
display(Markdown(f"## {decision['status']}"))
display(pd.DataFrame([decision['results']]))
display(pd.read_csv(OUT / 'barrier-summary.csv'))
display(pd.read_csv(OUT / 'probe-metrics.csv'))
for name in ['matching-response-curves.png','probe-family-pass-count.png','probe-predicts-full.png']:
    display(Image(filename=str(OUT / name)))


## Curate and publish

Publication defaults to enabled. Set `MINICELLS_PUBLISH=0` before this cell to keep results local.


In [ ]:
publish = os.environ.get('MINICELLS_PUBLISH', '1').strip().lower() not in {'0','false','no'}
cmd = [sys.executable, 'scripts/research/publish_experiment_019b_results.py']
if publish: cmd.append('--push')
subprocess.run(cmd, cwd=ROOT, check=True)
